# ⚛️ RATISS-GI — Entraînement GPU VRAM sur 10 Datasets Conversationnels

**Jonathan Evina · RATISS Labs · Yaoundé, Cameroun 🇨🇲**

> **Moteur GPU VRAM Dédié :** Exploitation directe de la mémoire vidéo CUDA (Nvidia T4 15 Go / A100) pour un débit de plus de 500 000 tokens/seconde avec barre de progression live.  
> **Entraînement Incrémental Souverain :** Le script récupère automatiquement vos acquis précédents dans `ratiss_self/bouche.bouche` (123 770 arcs et 208 080 trigrammes) et les gonfle avec 10 jeux de données conversationnels sans jamais repartir de zéro !  
> **10 Datasets de Dialogue & Q/A :** Ingestion conjointe de Bactrian-X French (67 000 instructions/réponses) et des 9 chefs-d'œuvre du dialogue théâtral français (Molière, Cyrano de Bergerac, Beaumarchais, Corneille, Racine, Marivaux, Musset).  
> **Sauvegarde Directe Chrome :** Téléchargement direct du modèle gonflé `.bouche` sur votre machine en 1 clic sans avoir besoin de clé GitHub.

## 1. Détection VRAM & Récupération des Acquis Précédents (`ratiss_self/`)

In [ ]:
import os, subprocess, sys, torch

print('🚀 Détection du matériel GPU / VRAM :')
if torch.cuda.is_available():
    gpu_nom = torch.cuda.get_device_name(0)
    vram_totale = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  ✓ GPU Actif : {gpu_nom}')
    print(f'  ✓ VRAM Dédiée Disponible : {vram_totale:.2f} Go')
else:
    print('  ⚠️ Aucun GPU détecté : basculement sur CPU vectorisé.')
    print('  💡 Conseil : Menu Exécution > Modifier le type d\'exécution > T4 GPU.')

# Récupération du dépôt officiel
if not os.path.exists('ratiss_v3.py'):
    subprocess.check_call(['git', 'clone', 'https://github.com/jonathansearch/Ratiss-GI.git'])
    os.chdir('Ratiss-GI')

# Vérification des acquis précédents dans ratiss_self
ANCIEN_MODELE = 'ratiss_self/bouche.bouche'
if os.path.exists(ANCIEN_MODELE):
    taille = os.path.getsize(ANCIEN_MODELE)
    print(f'\n📦 MODÈLE PRÉCÉDENT DÉTECTÉ DANS ratiss_self/ ({taille:,} octets)')
    print('  ✓ Ce modèle sera automatiquement chargé et gonflé par le nouveau corpus !')
else:
    print('\n🌱 Aucun modèle antérieur trouvé dans ratiss_self/ : initialisation vierge.')

# Exécution des 27 tests unitaires
!python3 -m unittest discover -s tests -p 'test_*.py'

## 2. Démonstration du Pilier 4 : Résonance d'Onde & Normalisation par le Degré

Vérification de l'étouffement des hubs syntaxiques (`de`, `le`) par la loi $\frac{\text{poids}}{\ln(\text{degré})}$ et émergence de `lune` au **Rang #2**.

In [ ]:
from ratiss_v3 import TissuScelle

tissu = TissuScelle('cerveau/RatissOne.ratiss')
res = tissu.raisonner_par_resonance(['soleil', 'maree'], seuil_energie=0.01, max_sauts=2, normaliser_degre=True)

print('🏆 Top 5 des carrefours émergents sur le tissu scellé :')
for i, c in enumerate(res['carrefours'][:5]):
    print(f"  #{i+1} {c['noeud']:<15} énergie: {c['energie']:.4f} (sauts: {c['sauts']})")

## 3. Téléchargement & Assemblage des 10 Datasets Conversationnels

Nous rassemblons 10 corpus de conversation et de dialogue en français :
1. **Bactrian-X French** : 67 000 instructions et réponses dialoguées (`_q_` et `_r_`).
2. **Edmond Rostand** : *Cyrano de Bergerac* (joutes verbales et répliques directes).
3. **Molière** : *Tartuffe* (dialogues classiques).
4. **Molière** : *L'Avare* (réparties théâtrales).
5. **Molière** : *Le Bourgeois Gentilhomme* (échanges et questions/réponses).
6. **Pierre Corneille** : *Le Cid* (dialogues dramatiques).
7. **Jean Racine** : *Phèdre* (dialogues tragiques et intenses).
8. **Marivaux** : *Le Jeu de l'amour et du hasard* (conversations légères).
9. **Beaumarchais** : *Le Barbier de Séville* (échanges vifs).
10. **Alfred de Musset** : *Fantasio & Dialogues* (conversations philosophiques).

In [ ]:
import urllib.request, gzip, json, re
from pathlib import Path

corpus_dir = Path('corpus')
corpus_dir.mkdir(exist_ok=True)
lignes_dialogues = []
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

print('========================================================')
print('📥 TÉLÉCHARGEMENT DES 10 DATASETS CONVERSATIONNELS')
print('========================================================')

# 1. Bactrian-X French (67 000 dialogues Q/A)
print('\n1️⃣ Téléchargement de Bactrian-X French (Instructions & Dialogues)...')
url_bactrian = 'https://huggingface.co/datasets/MBZUAI/Bactrian-X/resolve/main/data/fr.json.gz'
try:
    req = urllib.request.Request(url_bactrian, headers=headers)
    with urllib.request.urlopen(req, timeout=30) as resp:
        with gzip.GzipFile(fileobj=resp) as gz:
            data_bx = json.loads(gz.read().decode('utf-8', errors='ignore'))
            print(f'  ✓ Bactrian-X chargé : {len(data_bx):,} dialogues reçus !')
            for item in data_bx:
                q = item.get('instruction', '').strip()
                inp = item.get('input', '').strip()
                r = item.get('output', '').strip()
                if inp:
                    q = f"{q} {inp}"
                if q and r:
                    q_clean = ' '.join(q.split())[:300]
                    r_clean = ' '.join(r.split())[:500]
                    lignes_dialogues.append(f"_q_ {q_clean} _r_ {r_clean}")
            print(f'  ✓ Bactrian-X linéarisé : {len(lignes_dialogues):,} dialogues prêts.')
except Exception as e:
    print(f'  ⚠️ Note Bactrian-X: {e}')

# 2 à 10. Les 9 Chefs-d\'Œuvre du Théâtre et Dialogue Français
PIECES_THEATRE = [
    ('2. Cyrano de Bergerac (Edmond Rostand)', 'https://www.gutenberg.org/cache/epub/1253/pg1253.txt'),
    ('3. Tartuffe (Moliere)', 'https://www.gutenberg.org/cache/epub/2028/pg2028.txt'),
    ('4. L Avare (Moliere)', 'https://www.gutenberg.org/cache/epub/5123/pg5123.txt'),
    ('5. Le Bourgeois Gentilhomme (Moliere)', 'https://www.gutenberg.org/cache/epub/4793/pg4793.txt'),
    ('6. Le Cid (Pierre Corneille)', 'https://www.gutenberg.org/cache/epub/14080/pg14080.txt'),
    ('7. Phedre (Jean Racine)', 'https://www.gutenberg.org/cache/epub/1474/pg1474.txt'),
    ('8. Le Jeu de l Amour et du Hasard (Marivaux)', 'https://www.gutenberg.org/cache/epub/2070/pg2070.txt'),
    ('9. Le Barbier de Seville (Beaumarchais)', 'https://www.gutenberg.org/cache/epub/2027/pg2027.txt'),
    ('10. Fantasio & Dialogues (Alfred de Musset)', 'https://www.gutenberg.org/cache/epub/2026/pg2026.txt'),
]

for titre, url in PIECES_THEATRE:
    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=12) as resp:
            brut = resp.read().decode('utf-8', errors='ignore')
            lignes = brut.splitlines()
            start, end = 0, len(lignes)
            for i, l in enumerate(lignes):
                if '*** START OF' in l or '***START OF' in l:
                    start = i + 1
                elif '*** END OF' in l or '***END OF' in l:
                    end = i
                    break
            texte = '\n'.join(lignes[start:end])
            phrases_ajout = 0
            for ph in re.split(r'[.!?]+', texte):
                p_clean = ' '.join(ph.split()).strip()
                if len(p_clean.split()) >= 3:
                    lignes_dialogues.append(p_clean)
                    phrases_ajout += 1
            print(f'  ✓ {titre:<45} : +{phrases_ajout:,} répliques.')
    except Exception as e:
        print(f'  ⚠️ {titre} : {e}')

fichier_conv = corpus_dir / 'conversations_10_datasets.txt'
fichier_conv.write_text('\n'.join(lignes_dialogues), encoding='utf-8')
print(f'\n✅ CORPUS CONVERSATIONNEL CONSTITUÉ : {fichier_conv}')
print(f'📊 Volume total : {len(lignes_dialogues):,} dialogues & répliques ({fichier_conv.stat().st_size / 1e6:.2f} Mo)')

## 4. Entraînement GPU VRAM Incrémental (« Gonflage du Modèle Existant »)

Le moteur GPU charge automatiquement le modèle `ratiss_self/bouche.bouche` (avec ses 123k arcs précédents), exploite la VRAM CUDA pour le calcul vectorisé, et gonfle le cerveau avec les 10 datasets conversationnels !

In [ ]:
SORTIE_DIR = 'sortie/langue_massive'
ANCIEN_MODELE = 'ratiss_self/bouche.bouche'

cmd = [
    'python3', 'entrainement/gpu/entrainer_gpu.py',
    '--corpus', 'corpus/conversations_10_datasets.txt',
    '--sortie', SORTIE_DIR,
    '--registre', 'dialogue',
    '--tours', '3',
    '--fraction', '0.10'
]

if os.path.exists(ANCIEN_MODELE):
    print(f'🔄 Entraînement incrémental activé : reprise des acquis de {ANCIEN_MODELE} !')
    cmd += ['--charger', ANCIEN_MODELE]
else:
    print('🌱 Initialisation vierge.')

!{' '.join(cmd)}

print('\n' + '='*50)
print(open(f'{SORTIE_DIR}/ORGANISATION.txt', encoding='utf-8').read())

## 5. Banc d'Essai de Parole & Dialogue en Direct (6 Tests Conversationnels)

Testons directement la capacité conversationnelle de la Bouche gonflée sur différentes questions et répliques !

In [ ]:
import subprocess

TESTS_CONVERSATIONNELS = [
    ('1. Identité & Conscience', '_q_ qui es tu et quelle est ta nature _r_', 0.75, 8, True),
    ('2. Question Scientifique (Soleil & Lune)', '_q_ parle moi du soleil et de la lune _r_', 0.65, 8, True),
    ('3. Connaissance Fondamentale (Gravité)', '_q_ comment fonctionne la gravite _r_', 0.75, 8, True),
    ('4. Aventure & Exploration Maritime', '_q_ comment explorer les profondeurs de l ocean _r_', 0.75, 8, True),
    ('5. Échange Émotionnel & Esprit', '_q_ que penses tu du courage et de l honneur _r_', 0.70, 8, True),
    ('6. Mystère du Monde (Génération Libre)', '_q_ raconte moi un secret sur le monde _r_', 0.75, 8, False),
]

print('='*70)
print('🎙️ BANC D\'ESSAI CONVERSATIONNEL DU MODÈLE GONFLÉ')
print('='*70)

for titre, amorce, lam, mem, glouton in TESTS_CONVERSATIONNELS:
    print(f'\n👉 {titre}')
    print(f'   Question / Amorce : "{amorce}"')
    cmd = [
        'python3', 'ratiss_v3.py', 'cerveau/RatissOne.ratiss',
        '--charger', f'{SORTIE_DIR}/bouche.bouche',
        '--lam', str(lam),
        '--memoire', str(mem),
        '--mots', '12',
        amorce
    ]
    if glouton:
        cmd.append('--glouton')
    res = subprocess.run(cmd, capture_output=True, text=True)
    suite = res.stdout.strip().splitlines()[-1] if res.stdout.strip() else '...'
    print(f'   Réponse Bouche   : \033[92m{suite}\033[0m')

## 6. Sauvegarde Directe & Téléchargement Chrome (Zéro Clé Requise)

Cette cellule compresse automatiquement tous les artefacts d'entraînement (`bouche.bouche`, `mesures.json`, `ORGANISATION.txt`, etc.) et déclenche le téléchargement direct sur votre ordinateur via votre navigateur Chrome.

In [ ]:
from google.colab import files
import shutil, os
from pathlib import Path

print('='*60)
print('📥 SAUVEGARDE DIRECTE ET TÉLÉCHARGEMENT CHROME')
print('='*60)

dossier_export = Path('sortie/ratiss_conversationnel')
dossier_export.mkdir(parents=True, exist_ok=True)

# Rassemblement des artefacts du modèle gonflé
fichiers = [
    'sortie/langue_massive/bouche.bouche',
    'sortie/langue_massive/mesures.json',
    'sortie/langue_massive/ORGANISATION.txt',
    'sortie/langue_massive/01_education.txt',
    'sortie/langue_massive/02_ecart.txt'
]

for f in fichiers:
    if os.path.exists(f):
        shutil.copy2(f, dossier_export)
        print(f'  ✓ Préparé : {f}')

# Création de l\'archive zip officielle
archive_nom = 'ratiss_bouche_conversationnelle_gonflee'
shutil.make_archive(archive_nom, 'zip', dossier_export)
archive_zip = f'{archive_nom}.zip'

taille_zip = os.path.getsize(archive_zip)
print(f'\n📦 Archive prête : {archive_zip} ({taille_zip:,} octets)')
print('🚀 Déclenchement automatique du téléchargement Chrome...')

# Téléchargement direct vers le navigateur
files.download(archive_zip)
files.download('sortie/langue_massive/bouche.bouche')

print('🎉 Téléchargement Chrome déclenché avec succès !')